# Effect of the sampling frequency on the variance of the Sharpe ratio (symmetric case)

The variance of the Sharpe ratio of a GARCH process is 
$$ V = \dfrac1T \left[ 
  1 + \text{SR} ^2 \dfrac{  \kappa - 1 }{ 4 }
  \dfrac{
    ( 1 - \beta ) ^ 2 
  }{
    1 - \alpha^2 \kappa - 2 \alpha \beta - \beta^2
  }
  \dfrac{ 1 + \phi 
  }{ 
    1 - \phi
  }
\right] $$
where $\alpha$ is the ARCH parameter, $\beta$ the GARCH parameter, $\phi=\alpha+\beta$ the persistence,
SR the Sharpe ratio, $\kappa$ the non-excess kurtosis of the innovations (not of the returns), and $T$ the number of observations.
The formula is only valid if all the denominators are positive.

What happens if we change the sampling frequency? 
At high frequency, $T$ is larger, which drives the variance down, but the GARCH parameters are more extreme, which drives the variance up. Which effect dominates?

The following simulations show that $T$ matters much more.

In [ ]:
import ray
import numpy as np
import pandas as pd
import scipy
import matplotlib.pyplot as plt
from arch import arch_model
from tqdm.auto import tqdm
from functions import standardized_student, garch_returns, gjr_garch_returns, formula_15, estimate_parameters, standardized_jf_skew_t

def estimate_garch_parameters(r):
    "Estimate the GARCH parameters from the returns"
    if isinstance( r, pd.Series ):
        r = r.values
    T = len(r)
    mu = r.mean()
    y = r - mu
    sigma = y.std()
    SR = mu / sigma
    skew = scipy.stats.skew( y )  # Skewness of the returns

    model = arch_model(y, vol='Garch', p=1, q=1, dist='skewt', mean = 'Zero', rescale=True)
    res = model.fit(disp="off")
    innovations = res.resid / res.conditional_volatility
    alpha = res.params['alpha[1]']
    beta = res.params['beta[1]']
    phi = alpha + beta
    kurtosis = scipy.stats.kurtosis( innovations ) + 3  # Kurtosis of the innovations, not of the returns
    denominator = ( 1 - alpha**2 * kurtosis - 2 * alpha * beta - beta**2 )

    return { 
        'alpha': alpha,
        'beta': beta,
        'phi': phi,
        'SR': SR,
        'mu': mu,
        'sigma': sigma,
        'skew': skew,
        'kurtosis': kurtosis,
        'denominator': denominator,
        'T': T,
        'V': formula_15( SR, skew, kurtosis, alpha, beta, T ),
    }

from ray.util.multiprocessing import Pool
pool = Pool()
def tqdm_pool_imap_unordered( f, inputs ):
    return list( tqdm( pool.imap_unordered( f, inputs ), total = len(inputs) ) )

In [ ]:
def f( period, n = 1_000_000, R = 10 ): 

    r = []
    for iteration in range( R ): 
            
        n = 1_000_000
        df = 5
        sigma = .15
        mu = .08
        alpha = .1
        beta = .85
        kurtosis = 3 + 6 / ( df - 4 )

        innovations = standardized_student( size = n, df = df )
        ys, _ = garch_returns( 
            size = n, 
            mu = mu, sigma = sigma, alpha = alpha, beta = beta,
            innovations = innovations,
        )

        ys = pd.Series(ys).rolling(period).sum().dropna().values[::period] 
        tmp = estimate_garch_parameters( ys )
        tmp['period'] = period
        tmp['mu0'] = mu
        tmp['sigma0'] = sigma
        tmp['alpha0'] = alpha
        tmp['beta0'] = beta
        tmp['kurtosis0'] = kurtosis
        r.append( tmp )
    r = pd.DataFrame( r )
    return r

f(100)
r = tqdm_pool_imap_unordered( f, np.arange(1,200,1) )
r = pd.concat( r ).sort_values( 'period' ).reset_index( drop = True )
r['SR1'] = r['SR'] / np.sqrt( r['period'] )
r['V1'] = r['V'] / r['period']

In [ ]:
fig, axs = plt.subplots( 1, 2, figsize = (8,3), layout = 'constrained', dpi = 300 )
axs = axs.flatten()
s = 1
alpha = 1
x = r['period'] + np.random.uniform( -.5, .5, size = r.shape[0] )
axs[0].scatter( x, r['SR1'], s = s, alpha = alpha )
axs[0].axhline( r['mu0'].mean() / r['sigma0'].mean(), color = 'black', linestyle = ':', linewidth =1 )
axs[0].set_title( 'Rescaled Sharpe ratio' )
axs[1].scatter( x, r['V1'], s = s, alpha = alpha )
axs[1].set_ylim( 0, np.quantile( r['V1'], .95 ) )
axs[1].set_title( 'Theoretical variance of the rescaled Sharpe ratio' )
for ax in axs: 
    ax.set_xlabel( 'Period' )
plt.show()

In [ ]:
fig, axs = plt.subplots( 2, 3, figsize = (12,6), layout = 'constrained', dpi = 300 )
x = r['period'] + np.random.uniform( -.5, .5, size = r.shape[0] )
s = 1
alpha = 1
axs[0,0].scatter( x, r['alpha'], s = s, alpha = alpha )
axs[0,0].set_title( r'ARCH parameter $\alpha$' )
axs[0,1].scatter( x, r['beta'], s = s, alpha = alpha )
axs[0,1].set_title( r'GARCH parameter $\beta$' )
axs[0,2].scatter( x, r['phi'], s = s, alpha = alpha )
axs[0,2].axhline( 1, color = 'black', linestyle = ':', linewidth = 1 )
axs[0,2].set_title( r'Persistence $\phi = \alpha + \beta$' )

axs[1,0].scatter( x, r['skew'], s = s, alpha = alpha )
axs[1,0].axhline( 0, color = 'black', linestyle = ':', linewidth = 1 )
axs[1,0].set_ylim( -.5, .5 )
axs[1,0].set_title( 'Skewness of the returns' )

axs[1,1].scatter( x, r['kurtosis'], s = s, alpha = alpha )
axs[1,1].axhline( 3, color = 'black', linestyle = ':', linewidth = 1 )
axs[1,1].set_ylim( 2, 10 )
axs[1,1].set_title( 'Kurtosis of the innovations' )

axs[1,2].scatter( x, r['denominator'], s = s, alpha = alpha )
axs[1,2].set_title( r'Denominator $1 - \alpha^2 \kappa - 2 \alpha \beta - \beta^2$' )
axs[1,2].axhline( 0, color = 'black', linestyle = ':', linewidth = 1 )
axs[1,2].set_ylim( -.1, 1.04 )
for ax in axs.flatten(): 
    ax.set_xlabel( 'Period' )
plt.show()

# Asymmetric case

In the asymmetric case, the formula becomes
$$ V = \dfrac1T \left[ 
  1 
  - \text{SR} \dfrac{ 1 - \beta }{ 1 - \phi } s
  + \text{SR} ^2 \dfrac{  \kappa - 1 }{ 4 }
  \dfrac{
    ( 1 - \beta ) ^ 2 
  }{
    1 - \alpha^2 \kappa - 2 \alpha \beta - \beta^2
  }
  \dfrac{ 1 + \phi 
  }{ 
    1 - \phi
  }
\right] $$
where $s$ is the skewness of the returns (not of the innovations).

Negative skewness increases the variance. We ask the same question: what happens to the variance as we change the sampling frequency?

In [ ]:
def f( period, n = 1_000_000, R = 10 ): 

    r = []
    for iteration in range( R ): 

        n = 1_000_000
        #a, b = 12, 13.4  # Median, from real data
        a, b = 5, 20  # More extreme returns: skew = -1, kurt = 3 + 2.5
        sigma = .15
        mu = .08
        alpha = .1
        beta = .85

        innovations = standardized_jf_skew_t( size = n, a = a, b = b )
        ys, _ = garch_returns( 
            size = n, 
            mu = mu, sigma = sigma, alpha = alpha, beta = beta,
            innovations = innovations,
        )

        ys = pd.Series(ys).rolling(period).sum().dropna().values[::period] 
        tmp = estimate_garch_parameters( ys )
        tmp['period'] = period
        tmp['mu0'] = mu
        tmp['sigma0'] = sigma
        tmp['alpha0'] = alpha
        tmp['beta0'] = beta
        r.append( tmp )
        
    r = pd.DataFrame( r )
    return r

f(100)
r = tqdm_pool_imap_unordered( f, np.arange(1,200,1) )
r = pd.concat( r ).sort_values( 'period' ).reset_index( drop = True )
r['SR1'] = r['SR'] / np.sqrt( r['period'] )
r['V1'] = r['V'] / r['period']

In [ ]:
fig, axs = plt.subplots( 1, 2, figsize = (8,3), layout = 'constrained', dpi = 300 )
axs = axs.flatten()
s = 1
alpha = 1
x = r['period'] + np.random.uniform( -.5, .5, size = r.shape[0] )
axs[0].scatter( x, r['SR1'], s = s, alpha = alpha )
axs[0].axhline( r['mu0'].mean() / r['sigma0'].mean(), color = 'black', linestyle = ':', linewidth =1 )
axs[0].set_title( 'Rescaled Sharpe ratio' )
axs[1].scatter( x, r['V1'], s = s, alpha = alpha )
axs[1].set_ylim( 0, np.quantile( r['V1'], .95 ) )
axs[1].set_title( 'Theoretical variance of the rescaled Sharpe ratio' )
for ax in axs: 
    ax.set_xlabel( 'Period' )
plt.show()

In [ ]:
fig, axs = plt.subplots( 2, 3, figsize = (12,6), layout = 'constrained', dpi = 300 )
x = r['period'] + np.random.uniform( -.5, .5, size = r.shape[0] )
s = 1
alpha = 1
axs[0,0].scatter( x, r['alpha'], s = s, alpha = alpha )
axs[0,0].set_title( r'ARCH parameter $\alpha$' )
axs[0,1].scatter( x, r['beta'], s = s, alpha = alpha )
axs[0,1].set_title( r'GARCH parameter $\beta$' )
axs[0,2].scatter( x, r['phi'], s = s, alpha = alpha )
axs[0,2].axhline( 1, color = 'black', linestyle = ':', linewidth = 1 )
axs[0,2].set_title( r'Persistence $\phi = \alpha + \beta$' )

axs[1,0].scatter( x, r['skew'], s = s, alpha = alpha )
axs[1,0].axhline( 0, color = 'black', linestyle = ':', linewidth = 1 )
axs[1,0].set_ylim( -1.5, .1 )
axs[1,0].set_title( 'Skewness of the returns' )

axs[1,1].scatter( x, r['kurtosis'], s = s, alpha = alpha )
axs[1,1].axhline( 3, color = 'black', linestyle = ':', linewidth = 1 )
axs[1,1].set_ylim( 2, 10 )
axs[1,1].set_title( 'Kurtosis of the innovations' )

axs[1,2].scatter( x, r['denominator'], s = s, alpha = alpha )
axs[1,2].set_title( r'Denominator $1 - \alpha^2 \kappa - 2 \alpha \beta - \beta^2$' )
axs[1,2].axhline( 0, color = 'black', linestyle = ':', linewidth = 1 )
for ax in axs.flatten(): 
    ax.set_xlabel( 'Period' )
plt.show()